In [ ]:
%pip install -U gensim rapidfuzz

from pathlib import Path
from collections import Counter
import os
import re
import warnings

import pandas as pd

try:
    from gensim.models import Word2Vec
except ImportError as exc:
    raise ImportError(
        "کتابخانه gensim نصب نیست. در یک سلول جداگانه اجرا کنید: %pip install -U gensim"
    ) from exc

try:
    from rapidfuzz.distance import Levenshtein
except ImportError as exc:
    raise ImportError(
        "کتابخانه rapidfuzz نصب نیست. در یک سلول جداگانه اجرا کنید: %pip install -U rapidfuzz"
    ) from exc

INPUT_FILENAME = "train_data.csv"
MAX_REVIEWS_FOR_W2V = 150_000   # برای کاهش زمان آموزش؛ جست‌وجوی نهایی روی کل داده انجام می‌شود
WORD2VEC_VECTOR_SIZE = 100
WORD2VEC_WINDOW = 5
WORD2VEC_MIN_COUNT = 2
WORD2VEC_EPOCHS = 10
MAX_SIMILAR_TERMS_PER_SEED = 20
SIMILARITY_THRESHOLD = 0.40
MIN_TYPO_TOKEN_FREQUENCY = 2
MAX_EDIT_DISTANCE = 1
RANDOM_STATE = 42

# مسیر فایل: ابتدا پوشه جاری و سپس /mnt/data بررسی می‌شود.
def locate_input(filename: str) -> Path:
    candidates = [Path.cwd() / filename, Path("/mnt/data") / filename]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"فایل {filename} پیدا نشد. آن را کنار نوت‌بوک قرار دهید یا INPUT_FILENAME را اصلاح کنید."
    )

input_path = locate_input(INPUT_FILENAME)
output_dir = input_path.parent
print("فایل ورودی:", input_path)
print("پوشه خروجی:", output_dir)

فایل ورودی: d:\Downloads\train_data.csv
پوشه خروجی: d:\Downloads


In [ ]:

df = pd.read_csv(
    input_path,
    encoding="utf-8-sig",
    dtype={"asin": "string"},
    low_memory=False,
)

print("ابعاد دیتاست:", df.shape)
print("ستون‌ها:", df.columns.tolist())

# در فایل خام، reviewText متن اصلی و summary خلاصه نظر است.
# در فایل خروجی پیش‌پردازش قبلی، text_for_model از قبل ساخته شده است.
if "text_for_model" in df.columns:
    TEXT_COLUMN = "text_for_model"
    df["_search_text"] = df[TEXT_COLUMN].fillna("").astype(str)
elif "reviewText" in df.columns:
    TEXT_COLUMN = "reviewText"
    review_body = df["reviewText"].fillna("").astype(str)
    if "summary" in df.columns:
        review_summary = df["summary"].fillna("").astype(str)
        df["_search_text"] = (review_summary.str.strip() + ". " + review_body.str.strip()).str.strip(" .")
    else:
        df["_search_text"] = review_body
elif "review_text_clean" in df.columns:
    TEXT_COLUMN = "review_text_clean"
    df["_search_text"] = df[TEXT_COLUMN].fillna("").astype(str)
else:
    raise ValueError(
        "ستون متن پیدا نشد. یکی از ستون‌های text_for_model، reviewText یا review_text_clean لازم است."
    )

required_columns = {"asin", "overall"}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"ستون‌های ضروری موجود نیستند: {sorted(missing_columns)}")

# شناسه محصول و امتیاز را استاندارد می‌کنیم.
df["asin"] = df["asin"].astype("string").str.strip()
df["overall"] = pd.to_numeric(df["overall"], errors="coerce")

# رکوردهایی که شناسه یا متن ندارند برای این تحلیل قابل استفاده نیستند.
df = df[df["asin"].notna() & df["asin"].ne("")].copy()
df["_search_text"] = df["_search_text"].fillna("").astype(str)

print("ستون متن انتخاب‌شده:", TEXT_COLUMN)
print("تعداد ردیف‌های قابل بررسی:", f"{len(df):,}")
print("تعداد امتیازهای نامعتبر/خالی:", f"{df['overall'].isna().sum():,}")
display(df[["asin", "overall", "_search_text"]].head(3))

ابعاد دیتاست: (838944, 11)
ستون‌ها: ['overall', 'vote', 'verified', 'reviewTime', 'reviewerID', 'asin', 'style', 'reviewerName', 'reviewText', 'summary', 'unixReviewTime']
ستون متن انتخاب‌شده: reviewText
تعداد ردیف‌های قابل بررسی: 838,944
تعداد امتیازهای نامعتبر/خالی: 0


,asin,overall,_search_text
0,0511189877,2,Cannot Learn. I have an older URC-WR7 remote a...
1,0511189877,5,zero programming needed! Miracle!?. First tim...
2,0511189877,4,Works Good and programs easy.. Got them and on...


In [3]:
# توکن‌سازی متن انگلیسی برای Word2Vec و جست‌وجوی واژه‌ها
TOKEN_PATTERN = re.compile(r"[a-z]+(?:'[a-z]+)?")

def tokenize(text: str) -> list[str]:
    """استخراج توکن‌های انگلیسی با تبدیل حروف به حالت کوچک."""
    if not isinstance(text, str):
        return []
    return TOKEN_PATTERN.findall(text.lower())

# برای آموزش Word2Vec ممکن است از نمونه‌ای از نظرات استفاده کنیم تا مصرف حافظه/زمان کنترل شود.
embedding_texts = df["_search_text"].dropna().astype(str)
if MAX_REVIEWS_FOR_W2V is not None and len(embedding_texts) > MAX_REVIEWS_FOR_W2V:
    embedding_texts = embedding_texts.sample(n=MAX_REVIEWS_FOR_W2V, random_state=RANDOM_STATE)

sentences = [tokens for text in embedding_texts for tokens in [tokenize(text)] if len(tokens) >= 2]
if not sentences:
    raise ValueError("متن کافی برای آموزش Word2Vec پیدا نشد.")

print(f"تعداد نظرات استفاده‌شده برای آموزش Word2Vec: {len(sentences):,}")
print(f"تعداد کل توکن‌های نمونه آموزشی: {sum(map(len, sentences)):,}")

workers = max(1, min(8, (os.cpu_count() or 2) - 1))
word2vec_model = Word2Vec(
    sentences=sentences,
    vector_size=WORD2VEC_VECTOR_SIZE,
    window=WORD2VEC_WINDOW,
    min_count=WORD2VEC_MIN_COUNT,
    workers=workers,
    sg=1,
    negative=10,
    epochs=WORD2VEC_EPOCHS,
    seed=RANDOM_STATE,
)

print("تعداد واژه‌های یادگرفته‌شده:", len(word2vec_model.wv))

تعداد نظرات استفاده‌شده برای آموزش Word2Vec: 150,000
تعداد کل توکن‌های نمونه آموزشی: 18,307,905
تعداد واژه‌های یادگرفته‌شده: 43957


In [4]:
# استخراج واژه‌های نزدیک به warranty و guarantee با Word2Vec
SEED_WORDS = ["warranty", "guarantee"]
semantic_rows = []
semantic_terms = set()

for seed in SEED_WORDS:
    if seed not in word2vec_model.wv:
        warnings.warn(
            f"واژه {seed!r} در واژگان Word2Vec وجود ندارد؛ برای آن واژه مشابه استخراج نشد. "
            "در این حالت عبارت‌های اصلی و غلط‌های املایی همچنان جست‌وجو می‌شوند."
        )
        continue

    for similar_word, similarity in word2vec_model.wv.most_similar(
        seed, topn=MAX_SIMILAR_TERMS_PER_SEED
    ):
        semantic_rows.append({
            "seed_word": seed,
            "candidate_word": similar_word,
            "similarity": float(similarity),
            "selected_by_threshold": bool(similarity >= SIMILARITY_THRESHOLD),
        })
        if similarity >= SIMILARITY_THRESHOLD:
            semantic_terms.add(similar_word)

semantic_candidates_df = pd.DataFrame(
    semantic_rows,
    columns=["seed_word", "candidate_word", "similarity", "selected_by_threshold"],
)

print("واژه‌های مشابه پیشنهادی:")
if not semantic_candidates_df.empty:
    display(semantic_candidates_df.sort_values("similarity", ascending=False))
else:
    print("مدل برای دو واژه اصلی، کاندیدایی تولید نکرد.")

# این فهرست را می‌توانید پس از بررسی خروجی بالا دستی تغییر دهید.
# فقط واژه‌هایی را اضافه کنید که واقعاً در بافت ضمانت کالا کاربرد دارند.
MANUAL_EXTRA_TERMS = [
    # مثال: "coverage", "covered", "replacement", "protection"
]

# واژه‌های عمومی یا نامرتبط را در صورت مشاهده در خروجی Word2Vec در این فهرست قرار دهید.
EXCLUDED_SEMANTIC_TERMS = []
semantic_terms.update(tokenize(" ".join(MANUAL_EXTRA_TERMS)))
semantic_terms.difference_update(tokenize(" ".join(EXCLUDED_SEMANTIC_TERMS)))

print("تعداد واژه‌های معنایی انتخاب‌شده:", len(semantic_terms))

واژه‌های مشابه پیشنهادی:


,seed_word,candidate_word,similarity,selected_by_threshold
0,warranty,warrantee,0.839244,True
1,warranty,warrenty,0.813350,True
2,warranty,expired,0.779599,True
3,warranty,expires,0.760688,True
4,warranty,honor,0.747967,True
5,warranty,honors,0.732413,True
6,warranty,policy,0.731363,True
7,warranty,warranties,0.729763,True
8,warranty,voided,0.720560,True
20,guarantee,guaranty,0.717612,True


تعداد واژه‌های معنایی انتخاب‌شده: 34


In [5]:
# شناسایی غلط‌های املایی با فاصله ویرایشی (Levenshtein)
# فهرست زیر چند شکل رایج یا محتمل از غلط‌نویسی این دو واژه را پوشش می‌دهد.
COMMON_MISSPELLINGS = {
    "warrenty", "waranty", "warrany", "warrenty", "waranties",
    "guarentee", "garantee", "guarente", "garanty", "guarenteed",
}

# واژگان را از همان نمونه‌ای که برای آموزش Word2Vec استفاده شد استخراج می‌کنیم.
token_frequencies = Counter(token for sentence in sentences for token in sentence)

# غلط‌های املاییِ دیده‌شده در دیتاست که حداکثر یک ویرایش با واژه اصلی فاصله دارند.
typo_terms = set(COMMON_MISSPELLINGS)
for token, frequency in token_frequencies.items():
    if frequency < MIN_TYPO_TOKEN_FREQUENCY or len(token) < 6:
        continue
    if token in SEED_WORDS:
        continue
    distance = min(Levenshtein.distance(token, seed) for seed in SEED_WORDS)
    if distance <= MAX_EDIT_DISTANCE:
        typo_terms.add(token)

# شکل‌های صرفی مهم و عبارت‌های چندکلمه‌ای را نیز جداگانه اضافه می‌کنیم.
BASE_TERMS = {
    "warranty", "warranties", "warrant", "warranted",
    "guarantee", "guarantees", "guaranteed", "guaranteeing",
}

# عبارت‌های چندکلمه‌ای با دقت نسبتاً بالا؛ عمداً واژه‌های بسیار عمومی مثل service به‌تنهایی نیامده‌اند.
PHRASE_TERMS = {
    "extended warranty",
    "manufacturer warranty",
    "limited warranty",
    "lifetime warranty",
    "extended guarantee",
    "money back guarantee",
    "money back warranty",
    "protection plan",
    "service contract",
    "warranty period",
}

print("غلط‌های املایی/نزدیک‌یافته‌شده:", sorted(typo_terms))
print("واژه‌های پایه:", sorted(BASE_TERMS))
print("عبارت‌های چندکلمه‌ای:", sorted(PHRASE_TERMS))

غلط‌های املایی/نزدیک‌یافته‌شده: ['garantee', 'garanty', 'guaranteed', 'guarantees', 'guarente', 'guarentee', 'guarenteed', 'guarrantee', 'gurantee', 'waranties', 'waranty', 'warrant', 'warrants', 'warrany', 'warrenty']
واژه‌های پایه: ['guarantee', 'guaranteed', 'guaranteeing', 'guarantees', 'warrant', 'warranted', 'warranties', 'warranty']
عبارت‌های چندکلمه‌ای: ['extended guarantee', 'extended warranty', 'lifetime warranty', 'limited warranty', 'manufacturer warranty', 'money back guarantee', 'money back warranty', 'protection plan', 'service contract', 'warranty period']


In [ ]:
# ساخت واژگان نهایی و پیدا کردن نظرات مرتبط با ضمانت
single_word_terms = set(BASE_TERMS) | set(semantic_terms) | set(typo_terms)

# همه عبارت‌های چندکلمه‌ای را به توکن تبدیل می‌کنیم تا فاصله و خط تیره در متن مشکلی ایجاد نکند.
phrase_token_sequences = [tuple(tokenize(phrase)) for phrase in PHRASE_TERMS]
phrase_token_sequences = [seq for seq in phrase_token_sequences if len(seq) > 1]

def find_warranty_matches(text: str) -> list[str]:
    tokens = tokenize(text)
    token_set = set(tokens)
    found = token_set.intersection(single_word_terms)

    # جست‌وجوی عبارت‌های چندکلمه‌ای در توالی توکن‌ها
    for phrase_tokens in phrase_token_sequences:
        size = len(phrase_tokens)
        if any(tuple(tokens[i:i + size]) == phrase_tokens for i in range(len(tokens) - size + 1)):
            found.add(" ".join(phrase_tokens))

    return sorted(found)

df["matched_terms"] = df["_search_text"].map(find_warranty_matches)
df["has_warranty_discussion"] = df["matched_terms"].map(bool)

warranty_reviews = df[df["has_warranty_discussion"]].copy()
warranty_reviews["matched_terms"] = warranty_reviews["matched_terms"].map(lambda terms: "; ".join(terms))

print(f"تعداد کل نظرات: {len(df):,}")
print(f"تعداد نظرات مرتبط با ضمانت: {len(warranty_reviews):,}")
print(f"درصد نظرات مرتبط: {len(warranty_reviews) / len(df) * 100:.2f}%" if len(df) else "داده‌ای وجود ندارد.")

if warranty_reviews.empty:
    raise ValueError(
        "هیچ نظری پیدا نشد. SIMILARITY_THRESHOLD را بازبینی کنید، "
        "یا MANUAL_EXTRA_TERMS و PHRASE_TERMS را متناسب با دیتاست گسترش دهید."
    )

display(warranty_reviews[["asin", "overall", "matched_terms", "_search_text"]].head(10))

In [ ]:
# محاسبه میانگین overall برای هر محصول
# هر ردیف از warranty_reviews یک نظر مرتبط با ضمانت است.
product_summary = (
    warranty_reviews
    .groupby("asin", dropna=False)
    .agg(
        warranty_review_count=("asin", "size"),
        rated_warranty_review_count=("overall", "count"),
        mean_overall=("overall", "mean"),
        median_overall=("overall", "median"),
    )
    .reset_index()
)

# در صورت وجود عنوان و برند در فایل ورودی، آن‌ها را نیز به گزارش اضافه می‌کنیم.
for optional_col in ["brand", "product_title", "title"]:
    if optional_col in warranty_reviews.columns:
        metadata = (
            warranty_reviews[["asin", optional_col]]
            .dropna(subset=[optional_col])
            .drop_duplicates(subset=["asin"])
        )
        product_summary = product_summary.merge(metadata, on="asin", how="left")

# اگر فایل عنوان/برند محصولات از پروژه قبلی کنار train_data.csv باشد،
# نام محصول و برند را نیز به گزارش اضافه می‌کنیم. وجود این فایل اجباری نیست.
product_info_path = output_dir / "title_brand.csv"
if product_info_path.is_file():
    product_info = pd.read_csv(
        product_info_path,
        encoding="utf-8-sig",
        dtype={"asin": "string"},
        low_memory=False,
    )
    if {"asin", "title", "brand"}.issubset(product_info.columns):
        product_info["asin"] = product_info["asin"].astype("string").str.strip()
        product_info = product_info.drop_duplicates(subset=["asin"])[["asin", "title", "brand"]]
        product_info = product_info.rename(columns={"title": "product_title"})
        for col in ["product_title", "brand"]:
            if col not in product_summary.columns:
                product_summary = product_summary.merge(
                    product_info[["asin", col]], on="asin", how="left"
                )

product_summary = product_summary.sort_values(
    ["mean_overall", "warranty_review_count"],
    ascending=[False, False],
    na_position="last",
).reset_index(drop=True)

print("تعداد محصولات دارای حداقل یک نظر مرتبط با ضمانت:", f"{len(product_summary):,}")
print("خلاصه امتیاز نظرات مرتبط با ضمانت:")
display(product_summary.head(20))

# آمار کلی نظرات مرتبط با ضمانت
rated_overall = warranty_reviews["overall"].dropna()
if not rated_overall.empty:
    print(f"میانگین overall در تمام نظرات مرتبط: {rated_overall.mean():.3f}")
    print(f"تعداد نظرات مرتبط دارای امتیاز: {len(rated_overall):,}")
else:
    print("هیچ امتیاز معتبری در نظرات مرتبط با ضمانت وجود ندارد.")

In [ ]:
# ذخیره خروجی‌ها
reviews_output_path = output_dir / "warranty_related_reviews.csv"
summary_output_path = output_dir / "warranty_satisfaction_by_product.csv"
terms_output_path = output_dir / "warranty_related_terms.csv"

# ستون‌های کمکی داخلی را از خروجی نظرات حذف می‌کنیم؛ matched_terms حفظ می‌شود.
warranty_reviews_export = warranty_reviews.drop(
    columns=["_search_text", "has_warranty_discussion"], errors="ignore"
).copy()

warranty_reviews_export.to_csv(reviews_output_path, index=False, encoding="utf-8-sig")
product_summary.to_csv(summary_output_path, index=False, encoding="utf-8-sig")

term_rows = []
for term in sorted(BASE_TERMS):
    term_rows.append({"term": term, "source": "base_term"})
for term in sorted(semantic_terms):
    term_rows.append({"term": term, "source": "word2vec_or_manual"})
for term in sorted(typo_terms):
    term_rows.append({"term": term, "source": "spelling_variant_or_fuzzy_match"})
for term in sorted(PHRASE_TERMS):
    term_rows.append({"term": term, "source": "multiword_phrase"})

terms_df = pd.DataFrame(term_rows).drop_duplicates()
terms_df.to_csv(terms_output_path, index=False, encoding="utf-8-sig")

print("فایل نظرات مرتبط:", reviews_output_path)
print("فایل میانگین امتیاز هر محصول:", summary_output_path)
print("فایل واژه‌های استفاده‌شده:", terms_output_path)
print("\nنکته: برای نتیجه دقیق‌تر، جدول کاندیداهای Word2Vec و واژه‌های منتخب را از نظر معنایی بازبینی کنید.")